In [ ]:
import numpy as np
import math
from scipy.special import jv, jn_zeros
from pymittagleffler import mittag_leffler
import matplotlib.pyplot as plt
import time
from scipy.stats import levy_stable

In [ ]:
# Coefficients c_n with f(y) = (1-y^2/R^2)^m
def c_n(n,m):
    zeros = jn_zeros(0, n)[-1] # j_{0,n}
    return math.factorial(m) * (2/zeros)**(m+1) * jv(m+1, zeros) / jv(1, zeros)**2

# "practically" exact solution
def u(t,r,α,R,m,N_max,ϵ):
    prev_term = None
    sum = 0
    for n in range(1,N_max+1):
        λ = jn_zeros(0, n)[-1] / R
        exponent = t**α * λ**2
        sum += c_n(n,m) * np.real( mittag_leffler(-exponent, α, beta=1.0) ) * jv(0, λ*r)
        
        if prev_term is not None:
            if abs(sum - prev_term) < ϵ:
                return sum, n  # converged at term n
        prev_term = sum

In [ ]:
# Inverse sub.
T = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
α, θ = 0.5, 1.0
σ = (θ * np.cos(np.pi * α / 2))**(1/α)
ρ, μ = 1, 0
N_samples = 50_000
η = levy_stable.rvs(α, ρ, loc=μ, scale=σ, size=N_samples)
LT = (T[0] / η)**α

# The method
def mc_euler(
    R=1.0,
    M=10_000,
    dt=1e-3,
    f=lambda r, R: (1- (r/R)**2)**3,
    seed=123,
    sample_LT=LT
):
    rng = np.random.default_rng(seed)
    X = np.zeros((M, 2))
    alive = np.ones(M, dtype=bool)
    contrib = np.zeros(M, dtype=float)

    for i in range(M):
        t_curr = 0.0
        while t_curr < LT[i] and alive[i]:
            # Paso de tiempo: dt hasta antes de LT, último paso = resto
            if t_curr + dt < LT[i]:
                use_dt = dt
            else:
                use_dt = LT[i] - t_curr  # último paso exacto

            # Incremento Browniano (generador Δ → varianza 2*dt por coordenada)
            Z = rng.normal(0.0, 1.0, size=2)
            X[i] += np.sqrt(2.0 * use_dt) * Z
            t_curr += use_dt

            # Absorción naive: si termina fuera, muere
            if np.linalg.norm(X[i]) >= R:
                alive[i] = False

        # Si sobrevivió hasta LT, aporta f(B_LT)
        if alive[i]:
            r_final = np.linalg.norm(X[i])
            contrib[i] = f(r_final, R)

    # Estimador Monte Carlo
    u_hat = contrib.mean()
    return u_hat